# 5. Pass execution at pace


### Purpose

Assess whether teams maintain pass completion as the pace of preceding puck movement increases. The coaching question is: **can this team execute its passes at high pace, and in which situations does execution break down?**

The global premise is that playing faster is useful when a team maintains execution and converts movement into productive advancement. This notebook addresses execution; entry and defensive outcomes remain separate questions.

### Expected outcome

A pass-level research table, source-specific pace quintiles, and completion comparisons for sources and teams, with context and sample coverage. Higher pace is not assumed to be better. These observational comparisons identify sequences for coaching review rather than prove that changing pace causes a change in completion.


## 5.1. Setup and load processed data

Load augmented events from notebook 02, transition pace from notebook 03, and reference tables. Retain the export manifest's timing assumptions. Pace measures puck movement before the pass, not pass-flight speed or player skating speed.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


,table,rows,columns
0,events,86670,27
1,transitions,63298,25
2,teams,20,3
3,games,34,5
4,players,435,4


Timing: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 5.2. Build one record per pass attempt

Use `Pass` and `Incomplete Pass` release rows; generated reception rows are not additional attempts. Successful receptions supply actual target coordinates, while incomplete receptions supply intended targets. Pass length therefore describes recorded or intended endpoint displacement, not a measured puck trajectory. Direct/indirect pass type is retained.

Keep all manpower situations and retain zone, score, team, player, and game context. Pair a release with its immediately following expanded reception using the processing convention in notebook 01, and validate that relationship.


In [2]:
passes = events.loc[events["event"].isin(["Pass", "Incomplete Pass"])].copy()
passes["complete"] = passes["event"].eq("Pass")
passes["pass_type"] = passes["detail_1"].fillna("Unknown")
passes["situation"] = (
    passes["team_skaters"].astype(str) + "v" + passes["opponent_skaters"].astype(str)
)
passes["team_name"] = passes["team_id"].map(teams.set_index("team_id")["team_name"])
passes["player_name"] = passes["player_id"].map(players.set_index("player_id")["player_name"])
passes["game_date"] = passes["game_id"].map(games.set_index("game_id")["game_date"])

receptions = events.loc[events["event"].isin(["Pass Reception", "Incomplete Pass Reception"])].copy()
receptions["release_event_id"] = receptions["event_id"] - 1
passes = passes.merge(
    receptions[["release_event_id", "game_id", "team_id", "event", "x", "y"]],
    left_on="event_id", right_on="release_event_id", suffixes=("", "_target"),
    how="left", validate="one_to_one",
)
assert passes["release_event_id"].notna().all()
assert passes["game_id"].eq(passes["game_id_target"]).all()
assert passes["team_id"].eq(passes["team_id_target"]).all()
assert passes["complete"].eq(passes["event_target"].eq("Pass Reception")).all()
passes["pass_length_ft"] = np.hypot(passes["x_target"] - passes["x"], passes["y_target"] - passes["y"])
passes["length_band"] = pd.cut(
    passes["pass_length_ft"].astype(float), bins=[0, 20, 40, np.inf],
    labels=["0-20 ft", "20-40 ft", "Over 40 ft"], include_lowest=True,
)


## 5.3. Calculate pace preceding each release

Use a five-second lookback within the same inferred possession sequence. Pool distance and modeled duration from eligible movements completed in `[release time - lookback, release time)`. End event IDs must also precede the pass release. This excludes the focal pass, its generated reception, and movement ending at the release itself.

At sequence starts, use available history; record its span. This is an adaptation of the paper's event-focused approach, not an exact replication. Only completed movements are used, and full allocations are retained when their ends fall inside the window.

Empty or zero-duration histories have **unknown preceding pace**, not zero, and are excluded from quintiles while remaining visible in coverage. Modeled fractions are estimates; results require timing sensitivity checks before coaching recommendations.


In [3]:
LOOKBACK_SECONDS = 5
assert LOOKBACK_SECONDS > 0
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()].copy()
# Small numeric arrays keep the per-pass calculation straightforward and fast.
movement_groups = {
    pid: group[["end_event_id", "end_modeled_clock_seconds", "distance_ft",
                "modeled_elapsed_seconds"]].to_numpy(dtype=float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
sequence_start = events.groupby("possession_id")["modeled_clock_seconds"].max()
pace_rows = []

for attempt in passes.itertuples(index=False):
    movements = movement_groups.get(attempt.possession_id)
    if movements is not None:
        # Clocks count down: earlier completed movements have larger clocks.
        movements = movements[
            (movements[:, 0] < attempt.event_id)
            & (movements[:, 1] > attempt.modeled_clock_seconds)
            & (movements[:, 1] <= attempt.modeled_clock_seconds + LOOKBACK_SECONDS)
        ]
    count = 0 if movements is None else len(movements)
    distance = 0.0 if movements is None else float(movements[:, 2].sum())
    duration = 0.0 if movements is None else float(movements[:, 3].sum())
    pace_rows.append({
        "event_id": attempt.event_id,
        "preceding_transitions": count,
        "preceding_distance_ft": distance,
        "preceding_movement_seconds": duration,
        "preceding_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "available_history_seconds": min(
            LOOKBACK_SECONDS, float(sequence_start.loc[attempt.possession_id] - attempt.modeled_clock_seconds)
        ),
        "pace_status": "eligible" if duration > 0 else "no_history" if count == 0 else "zero_duration",
    })

passes = passes.merge(pd.DataFrame(pace_rows), on="event_id", validate="one_to_one")
coverage = passes.groupby(["source_dataset", "pace_status"]).agg(
    attempts=("event_id", "size"), completion_rate=("complete", "mean")
).reset_index()
display(coverage)


,source_dataset,pace_status,attempts,completion_rate
0,NWHL,eligible,5997,0.68751
1,NWHL,no_history,4621,0.667821
2,NWHL,zero_duration,61,0.655738
3,Olympics 2022,eligible,2585,0.796518
4,Olympics 2022,no_history,1791,0.781128
5,Olympics 2022,zero_duration,16,0.875
6,Womens,eligible,5947,0.740878
7,Womens,no_history,4097,0.726385
8,Womens,zero_duration,61,0.688525


## 5.4. Define source-specific pace quintiles

Use eligible pass attempts to define five equal-frequency pace bands within each source. Share these boundaries across its teams, so Q5 has the same meaning for every team within a source. Keep identical pace values together; duplicate quantile edges may produce fewer than five bands. Never split ties arbitrarily.

These boundaries are descriptive and use all games in the source. Any later predictive evaluation would fit boundaries using training games only.


In [4]:
eligible_passes = passes.loc[passes["pace_status"].eq("eligible")].copy()
eligible_passes["pace_quintile"] = pd.Series(pd.NA, index=eligible_passes.index, dtype="Int64")
quintile_edges = []
for source, group in eligible_passes.groupby("source_dataset"):
    bins, edges = pd.qcut(group["preceding_pace_ft_s"], q=5, labels=False, retbins=True, duplicates="drop")
    eligible_passes.loc[group.index, "pace_quintile"] = (bins + 1).astype("Int64")
    for i in range(len(edges) - 1):
        quintile_edges.append({"source_dataset": source, "pace_quintile": i + 1,
                               "lower_pace_ft_s": edges[i], "upper_pace_ft_s": edges[i + 1]})
quintile_boundaries = pd.DataFrame(quintile_edges)
display(quintile_boundaries.round(2))


,source_dataset,pace_quintile,lower_pace_ft_s,upper_pace_ft_s
0,NWHL,1,0.00,11.82
1,NWHL,2,11.82,17.04
2,NWHL,3,17.04,22.15
3,NWHL,4,22.15,29.54
4,NWHL,5,29.54,299.73
5,Olympics 2022,1,0.00,12.02
6,Olympics 2022,2,12.02,17.41
7,Olympics 2022,3,17.41,22.66
8,Olympics 2022,4,22.66,30.22
9,Olympics 2022,5,30.22,252.93


## 5.5. Compare pass completion across pace bands

Show completion percentages, attempts, games, and preceding movement exposure. Bootstrap games within each source to describe uncertainty in the pooled completion rates; this preserves within-game dependence. Quintile boundaries remain fixed during resampling. Repeated teams and the limited game sample still constrain interpretation.

A declining completion curve suggests execution challenges worth reviewing, but differences in pass selection, pressure, zone, and manpower can explain the association. It does not establish a safe speed threshold.


In [5]:
completion_by_quintile = eligible_passes.groupby(
    ["source_dataset", "pace_quintile"], observed=True
).agg(
    attempts=("event_id", "size"), completed=("complete", "sum"),
    games=("game_id", "nunique"), completion_rate=("complete", "mean"),
    median_pace_ft_s=("preceding_pace_ft_s", "median"),
    mean_pass_length_ft=("pass_length_ft", "mean"),
    median_history_seconds=("available_history_seconds", "median"),
).reset_index()
completion_by_quintile["completion_pct"] = 100 * completion_by_quintile["completion_rate"]

BOOTSTRAP_SAMPLES = 500
rng = np.random.default_rng(42)
intervals = []
for source, group in eligible_passes.groupby("source_dataset"):
    source_games = games.loc[games["source_dataset"].eq(source), "game_id"]
    for quintile, band in group.groupby("pace_quintile"):
        counts = band.groupby("game_id")["complete"].agg(["sum", "size"]).reindex(source_games, fill_value=0)
        draws = rng.integers(0, len(counts), size=(BOOTSTRAP_SAMPLES, len(counts)))
        totals = counts.to_numpy()[draws].sum(axis=1)
        rates = totals[:, 0] / np.where(totals[:, 1] > 0, totals[:, 1], np.nan)
        lower, upper = np.nanquantile(rates, [0.025, 0.975]) * 100
        intervals.append({"source_dataset": source, "pace_quintile": quintile,
                          "lower_pct": lower, "upper_pct": upper})
completion_by_quintile = completion_by_quintile.merge(
    pd.DataFrame(intervals), on=["source_dataset", "pace_quintile"], validate="one_to_one"
)
display(completion_by_quintile.round(2))

sources = completion_by_quintile["source_dataset"].unique()
completion_fig = make_subplots(rows=1, cols=len(sources), subplot_titles=list(sources), shared_yaxes=True)
for col, source in enumerate(sources, start=1):
    summary = completion_by_quintile.loc[completion_by_quintile["source_dataset"].eq(source)]
    completion_fig.add_trace(go.Scatter(
        x=summary["pace_quintile"], y=summary["completion_pct"], mode="lines+markers",
        name=source, showlegend=False,
        error_y=dict(type="data", symmetric=False,
                     array=np.maximum(summary["upper_pct"] - summary["completion_pct"], 0),
                     arrayminus=np.maximum(summary["completion_pct"] - summary["lower_pct"], 0)),
        customdata=summary[["attempts", "games", "median_pace_ft_s"]].to_numpy(),
        hovertemplate=("Q%{x}<br>Completion: %{y:.1f}%<br>Attempts: %{customdata[0]}"
                       "<br>Games: %{customdata[1]}<br>Median pace: %{customdata[2]:.1f} ft/s<extra></extra>"),
    ), row=1, col=col)
completion_fig.update_layout(title=f"Can teams execute at high pace? | preceding {LOOKBACK_SECONDS}s",
                             template="plotly_white", height=450)
completion_fig.update_xaxes(title_text="Source pace quintile (Q5 = fastest)", dtick=1)
completion_fig.update_yaxes(title_text="Pass completion (%)", range=[0, 100], row=1, col=1)
completion_fig.show()


,source_dataset,pace_quintile,attempts,completed,games,completion_rate,median_pace_ft_s,mean_pass_length_ft,median_history_seconds,completion_pct,lower_pct,upper_pct
0,NWHL,1,1200,834,15,0.7,8.06,35.8,5.0,69.5,66.06,72.52
1,NWHL,2,1199,806,15,0.67,14.54,35.71,5.0,67.22,64.69,69.96
2,NWHL,3,1199,828,15,0.69,19.47,35.93,5.0,69.06,65.89,72.03
3,NWHL,4,1199,834,15,0.7,25.14,36.29,5.0,69.56,67.22,71.72
4,NWHL,5,1200,821,15,0.68,41.90,37.37,4.0,68.42,65.66,70.98
5,Olympics 2022,1,517,411,6,0.79,8.01,35.98,5.0,79.5,76.38,82.07
6,Olympics 2022,2,517,414,6,0.8,14.83,36.32,5.0,80.08,76.99,82.72
7,Olympics 2022,3,517,409,6,0.79,19.98,36.33,5.0,79.11,75.77,82.09
8,Olympics 2022,4,517,401,6,0.78,25.55,37.38,5.0,77.56,74.20,82.91
9,Olympics 2022,5,517,424,6,0.82,39.82,37.93,4.0,82.01,78.86,85.36


## 5.6. Check context and team differences

Compare completion within zone, manpower, direct/indirect pass type, and length band, retaining the same source-wide quintiles. These are separate descriptive splits, not a fully adjusted model. Small cells need caution. The team chart hides cells with fewer than 30 attempts; the full table retains them.

A useful review starts with a team whose high-pace completion differs from its source, checks whether pass selection explains the pattern, and inspects individual attempts. No team should be labeled unable to play quickly from a sparse cell.


In [6]:
context_tables = {}
for context in ["event_team_zone", "situation", "pass_type", "length_band"]:
    context_tables[context] = eligible_passes.groupby(
        ["source_dataset", context, "pace_quintile"], observed=True, dropna=False
    ).agg(attempts=("event_id", "size"), completion_rate=("complete", "mean")).reset_index()
    context_tables[context]["completion_pct"] = 100 * context_tables[context]["completion_rate"]
    print(context)
    display(context_tables[context].round(2))

team_completion = eligible_passes.groupby(
    ["source_dataset", "team_id", "team_name", "pace_quintile"], observed=True
).agg(attempts=("event_id", "size"), games=("game_id", "nunique"),
      completion_rate=("complete", "mean")).reset_index()
team_completion["completion_pct"] = 100 * team_completion["completion_rate"]
MIN_TEAM_ATTEMPTS = 30
team_plot_data = team_completion.copy()
team_plot_data.loc[team_plot_data["attempts"].lt(MIN_TEAM_ATTEMPTS), "completion_pct"] = np.nan
team_fig = px.line(
    team_plot_data, x="pace_quintile", y="completion_pct", color="team_name",
    facet_col="source_dataset", markers=True, hover_data=["attempts", "games"],
    labels={"pace_quintile": "Source pace quintile", "completion_pct": "Pass completion (%)",
            "team_name": "Team"},
    title=f"Team execution profiles | at least {MIN_TEAM_ATTEMPTS} attempts per point",
)
team_fig.update_layout(template="plotly_white", height=500)
team_fig.update_xaxes(dtick=1)
team_fig.update_yaxes(range=[0, 100])
team_fig.show()


event_team_zone


,source_dataset,event_team_zone,pace_quintile,attempts,completion_rate,completion_pct
0,NWHL,DZ,1,431,0.71,71.23
1,NWHL,DZ,2,307,0.61,60.91
2,NWHL,DZ,3,253,0.64,64.03
3,NWHL,DZ,4,291,0.63,63.23
4,NWHL,DZ,5,505,0.65,64.95
5,NWHL,NZ,1,124,0.65,65.32
6,NWHL,NZ,2,115,0.76,75.65
7,NWHL,NZ,3,132,0.66,65.91
8,NWHL,NZ,4,149,0.67,67.11
9,NWHL,NZ,5,237,0.68,67.93


situation


,source_dataset,situation,pace_quintile,attempts,completion_rate,completion_pct
0,NWHL,4v3,1,7,1.0,100.0
1,NWHL,4v3,2,10,0.9,90.0
2,NWHL,4v3,3,6,1.0,100.0
3,NWHL,4v3,4,4,1.0,100.0
4,NWHL,4v3,5,3,1.0,100.0
...,...,...,...,...,...,...
113,Womens,6v5,1,13,0.69,69.23
114,Womens,6v5,2,16,0.69,68.75
115,Womens,6v5,3,10,0.8,80.0
116,Womens,6v5,4,18,0.83,83.33


pass_type


,source_dataset,pass_type,pace_quintile,attempts,completion_rate,completion_pct
0,NWHL,Direct,1,769,0.71,71.13
1,NWHL,Direct,2,792,0.68,68.18
2,NWHL,Direct,3,864,0.71,70.72
3,NWHL,Direct,4,847,0.71,70.96
4,NWHL,Direct,5,816,0.69,69.49
5,NWHL,Indirect,1,431,0.67,66.59
6,NWHL,Indirect,2,407,0.65,65.36
7,NWHL,Indirect,3,335,0.65,64.78
8,NWHL,Indirect,4,352,0.66,66.19
9,NWHL,Indirect,5,384,0.66,66.15


length_band


,source_dataset,length_band,pace_quintile,attempts,completion_rate,completion_pct
0,NWHL,0-20 ft,1,244,0.64,64.34
1,NWHL,0-20 ft,2,255,0.61,61.18
2,NWHL,0-20 ft,3,223,0.69,69.06
3,NWHL,0-20 ft,4,215,0.65,65.12
4,NWHL,0-20 ft,5,216,0.68,68.06
5,NWHL,20-40 ft,1,520,0.69,68.85
6,NWHL,20-40 ft,2,510,0.68,67.84
7,NWHL,20-40 ft,3,535,0.67,67.48
8,NWHL,20-40 ft,4,540,0.69,68.7
9,NWHL,20-40 ft,5,504,0.68,68.25


## 5.7. Inspect passes and summarize coaching questions

Select a source, team, and quintile to inspect passes behind the summary. Review the preceding event sequence using `possession_id` and `event_id` in the augmented events. Intended incomplete-pass targets must not be interpreted as actual receptions.

Before recommending faster or slower play, check consistency across games, compare equivalent situations, and rerun with another lookback. A recorded-time sensitivity analysis is also needed because modeled within-second pace can depend on event density. The defensible takeaway is a context-specific execution pattern to review, not a causal instruction to change speed.


In [7]:
REVIEW_SOURCE = eligible_passes["source_dataset"].iloc[0]
REVIEW_TEAM_ID = int(eligible_passes.loc[eligible_passes["source_dataset"].eq(REVIEW_SOURCE), "team_id"].iloc[0])
REVIEW_QUINTILE = 5
review_passes = eligible_passes.loc[
    eligible_passes["source_dataset"].eq(REVIEW_SOURCE)
    & eligible_passes["team_id"].eq(REVIEW_TEAM_ID)
    & eligible_passes["pace_quintile"].eq(REVIEW_QUINTILE)
]
display(review_passes[[
    "game_id", "game_date", "period", "clock_seconds", "event_id", "possession_id",
    "player_name", "complete", "preceding_pace_ft_s", "preceding_movement_seconds",
    "preceding_transitions", "event_team_zone", "situation", "pass_type", "pass_length_ft",
]].head(20))


,game_id,game_date,period,clock_seconds,event_id,possession_id,player_name,complete,preceding_pace_ft_s,preceding_movement_seconds,preceding_transitions,event_team_zone,situation,pass_type,pass_length_ft
1,3,2021-01-23,1,1183.0,11,3,Meghan Lorence,True,35.175654,3.500000,3,OZ,5v5,Direct,13.341664
7,3,2021-01-23,1,1149.0,34,9,Nina Rodgers,True,87.200917,0.500000,2,NZ,5v5,Direct,22.627417
31,3,2021-01-23,1,1052.0,107,22,Winny Brodt-Brown,False,127.906216,0.500000,2,DZ,5v5,Indirect,50.0
70,3,2021-01-23,1,856.0,243,45,Stephanie Anderson,False,34.058773,0.500000,1,DZ,5v5,Indirect,42.201896
131,3,2021-01-23,1,504.0,471,96,Meghan Lorence,True,44.271887,0.500000,2,NZ,4v5,Indirect,61.008196
132,3,2021-01-23,1,501.0,474,96,Stephanie Anderson,False,59.025696,2.666667,5,OZ,4v5,Direct,12.649111
133,3,2021-01-23,1,499.0,477,96,Stephanie Anderson,True,34.464847,5.500000,5,OZ,4v5,Direct,51.156622
200,3,2021-01-23,2,1174.0,754,157,Stephanie Anderson,True,49.446827,3.000000,3,OZ,5v5,Indirect,36.769553
211,3,2021-01-23,2,1123.0,790,165,Haylea Schmid,True,38.052595,0.500000,2,NZ,5v5,Direct,18.601075
214,3,2021-01-23,2,1099.0,804,167,Jonna Curtis,True,31.631559,3.000000,1,OZ,5v5,Direct,23.021729
